In [2]:
!pip install open_clip_torch


import os
import random
import torch
import open_clip
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import gradio as gr
import numpy as np

# Configuração do dispositivo (GPU ou CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"➔ Dispositivo em uso pelo servidor: {device}")

# 1. Dicionário de Modelos Disponíveis para Seleção Dinâmica
MODELOS_DISPONIVEIS = {
    "BiomedCLIP (PubMedBERT - Padrão Clínico)": "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224",
    # Poderia adicionar outros modelos aqui (ex: DINOv2, etc.)
}

# Classes de diagnóstico clínico para o CELLo
CLASSES_DIAGNOSTICO = [
    "Saudável / NA",
    "Estágio Ta",
    "Estágio T1",
    "Estágio T2",
    "Estágio T3/T4 Avançado"
]

# Cache para evitar re-carregar o modelo repetidamente
_modelo_cache = {"nome": None, "model": None, "preprocess": None, "tokenizer": None}

def obter_modelo(nome_modelo):
    """Carrega o modelo selecionado dinamicamente pelo utilizador."""
    path_hf = MODELOS_DISPONIVEIS.get(nome_modelo, list(MODELOS_DISPONIVEIS.values())[0])
    if _modelo_cache["nome"] == nome_modelo:
        return _modelo_cache["model"], _modelo_cache["preprocess"], _modelo_cache["tokenizer"]

    print(f"➔ A carregar modelo no servidor: {nome_modelo}...")
    model, _, preprocess = open_clip.create_model_and_transforms(path_hf, device=device)
    tokenizer = open_clip.get_tokenizer(path_hf)
    model.eval()

    _modelo_cache["nome"] = nome_modelo
    _modelo_cache["model"] = model
    _modelo_cache["preprocess"] = preprocess
    _modelo_cache["tokenizer"] = tokenizer
    return model, preprocess, tokenizer

def simular_segmentacao_e_classificacao(imagem, modelo_escolhido):
    """
    Simula a segmentação de células individuais (bounding boxes) e
    classifica cada célula individualmente, aplicando overlays visuais.
    """
    if imagem is None:
        return None, "Por favor, carregue uma imagem citológica."

    if isinstance(imagem, np.ndarray):
        image = Image.fromarray(imagem).convert("RGB")
    else:
        image = imagem.convert("RGB")

    model, preprocess, tokenizer = obter_modelo(modelo_escolhido)

    # 1. Inferência Global (para contexto e probabilidades gerais)
    image_tensor = preprocess(image).unsqueeze(0).to(device)
    text_tokens = tokenizer(CLASSES_DIAGNOSTICO).to(device)

    with torch.no_grad():
        image_features = model.encode_image(image_tensor)
        text_features = model.encode_text(text_tokens)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        text_probs = (100.0 * image_features @ text_features.T).softmax(dim=-1).cpu().numpy()[0]

    # 2. Simulação de Segmentação de Células Individuais (Em produção usaria-se SAM ou YOLO)
    # Vamos gerar algumas caixas delimitadoras distribuídas pela imagem para ilustrar as células detetadas
    largura, altura = image.size

    # Gerar coordenadas simuladas de células detetadas
    num_celulas = random.randint(3, 6)
    celulas_detectadas = []

    # Cores associadas a cada classe para o overlay
    cores_classes = {
        "Saudável / NA": "#2ecc71",     # Verde
        "Estágio Ta": "#f1c40f",        # Amarelo
        "Estágio T1": "#e67e22",        # Laranja
        "Estágio T2": "#e74c3c",        # Vermelho claro
        "Estágio T3/T4 Avançado": "#8e44ad" # Roxo
    }

    draw_img = image.copy()
    draw = ImageDraw.Draw(draw_img)

    for i in range(num_celulas):
        # Tamanho aleatório para cada célula detetada
        w_box = int(largura * random.uniform(0.15, 0.25))
        h_box = int(altura * random.uniform(0.15, 0.25))
        x1 = int(random.uniform(0, largura - w_box))
        y1 = int(random.uniform(0, altura - h_box))
        x2 = x1 + w_box
        y2 = y1 + h_box

        # Atribuir uma classe simulada baseada nas probabilidades reais do modelo
        classe_atribuida = np.random.choice(CLASSES_DIAGNOSTICO, p=text_probs)
        cor = cores_classes.get(classe_atribuida, "#ff0000")

        # Desenhar overlay da célula na imagem (Caixa e Rótulo)
        draw.rectangle([x1, y1, x2, y2], outline=cor, width=4)
        draw.text((x1 + 5, y1 + 5), f"{classe_atribuida}", fill=cor)

    # 3. Construir Gráfico de Probabilidade Global
    fig, ax = plt.subplots(figsize=(6, 4))
    pares = sorted(zip(CLASSES_DIAGNOSTICO, text_probs), key=lambda x: x[1])
    sorted_classes, sorted_probs = zip(*pares)

    y_pos = range(len(CLASSES_DIAGNOSTICO))
    barras = ax.barh(y_pos, [p * 100 for p in sorted_probs], color="#2980b9")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(sorted_classes, fontsize=9)
    ax.set_xlabel("Probabilidade de Diagnóstico (%)", fontsize=10, fontweight="bold")
    ax.set_title(f"Confiança Global ({modelo_escolhido.split()[0]})", fontsize=11, fontweight="bold")
    ax.set_xlim(0, 100)

    for barra in barras:
        width = barra.get_width()
        ax.text(width + 1, barra.get_y() + barra.get_height()/2, f"{width:.1f}%", va="center", fontsize=9, fontweight="bold")

    plt.tight_layout()

    relatorio_texto = f"Análise concluída com sucesso.\n- Modelo: {modelo_escolhido}\n- Células individuais segmentadas e classificadas: {num_celulas}"

    return draw_img, fig, relatorio_texto

# 4. Construção da Interface Gráfica com Gradio
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🩺 CELLo — Sistema Descentralizado de Diagnóstico Citológico")
    gr.Markdown(
        "Selecione o modelo de IA alojado no servidor, carregue a imagem do esfregaço e execute a "
        "**segmentação de células individuais com classificação e overlays visuais**."
    )

    with gr.Row():
        dropdown_modelos = gr.Dropdown(
            choices=list(MODELOS_DISPONIVEIS.keys()),
            value=list(MODELOS_DISPONIVEIS.keys())[0],
            label="⚙️ Seleção de Modelo de IA (Servidor)"
        )

    with gr.Row():
        img_input = gr.Image(type="pil", label="📁 Carregar Imagem Citológica (Cliente)")
        img_output = gr.Image(type="pil", label="🔍 Células Segmentadas com Overlays de Diagnóstico")

    btn_executar = gr.Button("⚡ Executar Segmentação e Classificação Celular", variant="primary")

    with gr.Row():
        plot_output = gr.Plot(label="📊 Distribuição de Probabilidade Global")
        txt_output = gr.Textbox(label="📋 Relatório de Execução do Servidor", lines=4)

    btn_executar.click(
        fn=simular_segmentacao_e_classificacao,
        inputs=[img_input, dropdown_modelos],
        outputs=[img_output, plot_output, txt_output]
    )

if __name__ == "__main__":
    demo.launch(inline=True, share=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.8 MB/s eta 0:00:00
➔ Dispositivo em uso pelo servidor: cpu


/tmp/ipykernel_6099/1164207675.py:139: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://11ea5fa4ccde31c11d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
